In [2]:

import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


# ==========================
# OpenAI Configuration
# ==========================

config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)

MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)


df=pd.read_csv('../../data/df_n3.csv')


Using model: gpt-5.6-luna with reasoning effort: medium


CREATE BATCH JSON

In [7]:


import json
import time
import random
import pandas as pd
from collections import defaultdict
from collections import Counter


# =========================
# PROMPT BUILDING
# =========================

def build_dedup_messages(rows, shuffle=False):
    """
    rows: list of dicts for comments that share the SAME patch_id.
          each dict must have: comment_id, generated_comment, 
        #   category, severity,
          hunk, generated_context
    """
    code_patch = rows[0]
    comments = list(rows)
    if shuffle:
        random.shuffle(comments)

    comments_text = "\n\n".join(
        f'ID: {c["comment_id"]}\n'
        f'Comment: {c["generated_comment"]}'
        # f'Category: {c["category"]}\n'
        # f'Severity: {c["severity"]}\n'
        for c in comments
    )

    system_prompt = """
You are an expert software engineer reviewing a set of code-review comments that were independently generated by different models for the SAME code change.

INPUT:
You may receive:
- Pull request title
- Target file
- Related code hunks from the same file
- Surrounding code context, wrapped in <context> tags
- The code patch, wrapped in <patch> tags
- A list of generated review comments, each with its ID, category, and severity

Use the available context only to understand the patch and determine whether two comments refer to the same underlying issue. Do not group comments simply because they mention nearby code or similar identifiers.

TASK 1 - GROUPING:
Group comments that describe the SAME underlying issue, even if phrased differently or focused on different symptoms. Two comments belong in the same group ONLY IF a human reviewer would consider one redundant given the other.

Do NOT group comments together just because they:
- refer to the same lines, variables, or functions but raise different concerns (e.g. bug vs. refactoring suggestion).
- use similar wording but identify different root causes.

Every comment must appear in exactly one group. A comment with no duplicate forms its own group.

TASK 2 - SYNTHESIS:
For every group containing TWO OR MORE comments, generate a single "synthesis_comment" that represents their shared concern. It should:
- Capture only the common issue.
- Be concise.
- Read like a human review comment, not a summary of comments.
- Avoid copying any individual comment verbatim.

Do NOT include "synthesis_comment" for singleton groups.

OUTPUT (STRICT JSON ONLY):
{
  "groups": [
    {
      "comment_ids": ["<id1>", "<id2>"],
      "issue_type": "short label",
      "target": "variable/function/line",
      "synthesis_comment": "..."
    },
    {
      "comment_ids": ["<id3>"],
      "issue_type": "short label",
      "target": "..."
    }
  ]
}

Use ONLY the exact comment IDs provided. Do not invent, split, merge, or omit IDs.
"""
    user_prompt = ""

    if "pr_title" in code_patch and pd.notna(code_patch["pr_title"]):

        user_prompt += f"""
PULL REQUEST TITLE:
{code_patch["pr_title"]}

"""

    if "target_file" in code_patch and pd.notna(code_patch["target_file"]):

        user_prompt += f"""
TARGET FILE:
{code_patch["target_file"]}

"""
    if "relevant_same_file_code_hunks" in code_patch and pd.notna(code_patch["relevant_same_file_code_hunks"]):

        user_prompt += f"""
RELATED CODE HUNKS IN TARGET FILE:
<related_code_hunks_in_target_file>
{code_patch["relevant_same_file_code_hunks"]}
</related_code_hunks_in_target_file>

"""

    if "relevant_context" in code_patch and pd.notna(code_patch["relevant_context"]):

        if str(code_patch["relevant_context"]).strip():

            user_prompt += """
SURROUNDING CODE CONTEXT:
This is additional context extracted from the original file around the changed code to help you in problem identification.
<context>
"""

            user_prompt += code_patch["relevant_context"]
            user_prompt += "\n</context>\n\n"

    user_prompt += f"""
CODE PATCH:
<patch>
{code_patch["hunk"]}
</patch>

"""
    user_prompt += f"""
COMMENTS TO GROUP:
{comments_text}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]




N_CONSISTENCY_RUNS = 3


def build_batch_jsonl(df, output_path="artifacts/dedup_batch.jsonl",
                      n_runs=N_CONSISTENCY_RUNS):
    """
    Build a JSONL file containing one Batch API request per LLM call.

    Each line corresponds to one consistency run for one patch.

    For each patch:
        run 0 -> original comment order
        run 1 -> shuffled comments
        run 2 -> shuffled comments

    The generated JSONL can then be submitted to the OpenAI Batch API.
    """

    required_cols = [
        "comment_id",
        "generated_comment",
        "generation_system",
        "patch_id",
        "hunk"
    ]

    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    # Create output directory if needed
    import os
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)

    patch_groups = df.groupby("patch_id")

    total_requests = 0

    with open(output_path, "w", encoding="utf-8") as f:

        for patch_id, group in patch_groups:

            rows = group.to_dict("records")

            # Make sure optional fields exist
            for r in rows:
                r.setdefault("generated_context", "")

            # One request for each consistency run
            for run_idx in range(n_runs):

                # Run 0 keeps original order.
                # Later runs shuffle the comments.
                shuffle = run_idx > 0

                messages = build_dedup_messages(
                    rows,
                    shuffle=shuffle
                )

                batch_request = {
                    "custom_id": f"dedup_{patch_id}_run_{run_idx}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": MODEL_NAME,
                        "messages": messages,
                        "response_format": {
                            "type": "json_object"
                        }
                    }
                }

                # Add reasoning effort only if you use it
                if REASONING_EFFORT:
                    batch_request["body"]["reasoning_effort"] = REASONING_EFFORT
                else:
                    batch_request["body"]["temperature"] = 0.3

                # Write exactly ONE JSON object per line
                f.write(
                    json.dumps(
                        batch_request,
                        ensure_ascii=False
                    ) + "\n"
                )

                total_requests += 1

    print(f"[DONE] JSONL created: {output_path}")
    print(f"[DONE] Number of patches: {patch_groups.ngroups}")
    print(f"[DONE] Number of requests: {total_requests}")

    return output_path




In [8]:

build_batch_jsonl(
    df,
    output_path="dedup_all_models.jsonl"
)

[DONE] JSONL created: dedup_all_models.jsonl
[DONE] Number of patches: 504
[DONE] Number of requests: 1512


'dedup_all_models.jsonl'

SUBLIT BATCH

In [9]:

 
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


INPUT_FILE = "dedup_all_models.jsonl"


print("[1/2] Uploading JSONL...")

with open(INPUT_FILE, "rb") as f:
    batch_file = client.files.create(
        file=f,
        purpose="batch"
    )

print(f"[UPLOAD] File ID: {batch_file.id}")


print("[2/2] Creating Batch...")

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"[BATCH] Batch ID: {batch.id}")
print(f"[BATCH] Status: {batch.status}")


[1/2] Uploading JSONL...
[UPLOAD] File ID: file-1iLR8xfJQhquyRyamcFmsA
[2/2] Creating Batch...
[BATCH] Batch ID: batch_6a98a49172688190a4638a476c9ad77f
[BATCH] Status: validating


VERIFY STATE OF BATCH SUBMISSION

In [3]:

from openai import OpenAI
from dotenv import load_dotenv



# Use the batch ID returned when you submitted the batch
# BATCH_ID = batch.id
BATCH_ID = "batch_6a98a49172688190a4638a476c9ad77f"

current_batch = client.batches.retrieve(BATCH_ID)

print("Batch ID:", current_batch.id)
print("Status:", current_batch.status)

if current_batch.request_counts:
    print("Total requests:", current_batch.request_counts.total)
    print("Completed:", current_batch.request_counts.completed)
    print("Failed:", current_batch.request_counts.failed)

if current_batch.output_file_id:
    print("Output file ID:", current_batch.output_file_id)

if current_batch.error_file_id:
    print("Error file ID:", current_batch.error_file_id)
    
    

Batch ID: batch_6a98a49172688190a4638a476c9ad77f
Status: completed
Total requests: 1512
Completed: 1512
Failed: 0
Output file ID: file-9yMwoDdhPsYZSjxLfJSuva


GET BATCH SUBMISSION RESULTS

In [12]:

OUTPUT_FILE = "dedup_batch_results.jsonl"

if current_batch.status != "completed":
    print(f"Batch is not finished yet. Current status: {current_batch.status}")
else:
    if current_batch.output_file_id is None:
        print("Batch completed but no output file was found.")
    else:
        print(f"Downloading output file: {current_batch.output_file_id}")

        result = client.files.content(
            current_batch.output_file_id
        )

        with open(OUTPUT_FILE, "wb") as f:
            f.write(result.read())

        print(f"[DONE] Results saved to: {OUTPUT_FILE}")


[DONE] Results saved to: dedup_batch_results.jsonl


GET JSON RESULTS and transform them to df_n4

In [13]:
import json
import random
from collections import defaultdict, Counter

import pandas as pd



BATCH_RESULTS_PATH = "dedup_batch_results.jsonl"

N_CONSISTENCY_RUNS = 3


def load_batch_results(jsonl_path):
    """
    Load OpenAI batch results from a JSONL file.

    Returns:
        list of raw batch result objects
    """

    results = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            results.append(json.loads(line))

    print(f"[BATCH] Loaded {len(results)} batch results")

    return results


def extract_batch_response(batch_result):


    try:
        response = batch_result["response"]

        body = response["body"]

        choices = body["choices"]

        if not choices:
            return None

        return choices[0]["message"]["content"]

    except Exception as e:
        print(f"[WARN] Could not extract response: {e}")
        return None


# ============================================================
# 3. GET CUSTOM ID
# ============================================================

def get_custom_id(batch_result):
    """
    Return the custom_id used when creating the batch request.

    Change this function ONLY if your batch custom_id has a
    particular structure.
    """

    return batch_result.get("custom_id")


# ============================================================
# 4. PARSE + VALIDATE LLM OUTPUT
# ============================================================

def parse_dedup_output(text, valid_ids):
  
    if text is None:
        return None

    # Sometimes models return ```json ... ```
    # even though JSON only was requested.
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "", 1)
        text = text.replace("```", "")
        text = text.strip()

    try:
        data = json.loads(text)
        raw_groups = data.get("groups", [])

    except Exception:
        return None

    valid_ids = set(valid_ids)

    seen = set()
    groups = []

    for g in raw_groups:

        ids = g.get("comment_ids", [])

        clean_ids = [
            i
            for i in ids
            if i in valid_ids and i not in seen
        ]

        if not clean_ids:
            continue

        seen.update(clean_ids)

        synthesis = (
            g.get("synthesis_comment")
            if len(clean_ids) > 1
            else None
        )

        if isinstance(synthesis, str):
            synthesis = synthesis.strip() or None

        groups.append(
            {
                "comment_ids": clean_ids,
                "synthesis_comment": synthesis,
            }
        )

    # Add comments that the LLM forgot
    missing = valid_ids - seen

    for m in missing:
        groups.append(
            {
                "comment_ids": [m],
                "synthesis_comment": None,
            }
        )

    return groups


# ============================================================
# 5. GROUPS -> PAIRS
# ============================================================

def groups_to_pairs(groups):
    """
    Convert groups into pairs of comments that co-occurred
    in the same group.
    """

    pairs = set()

    for g in groups:

        ids = g["comment_ids"]

        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):

                pairs.add(
                    frozenset(
                        (ids[i], ids[j])
                    )
                )

    return pairs


# ============================================================
# 6. UNION-FIND
# ============================================================

class UnionFind:

    def __init__(self, ids):

        self.parent = {
            i: i
            for i in ids
        }

    def find(self, x):

        while self.parent[x] != x:

            self.parent[x] = self.parent[
                self.parent[x]
            ]

            x = self.parent[x]

        return x

    def union(self, a, b):

        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[ra] = rb


# ============================================================
# 7. SELECT SYNTHESIS COMMENT
# ============================================================

def select_synthesis_for_cluster(
    cluster_ids,
    all_run_groups
):

    target = set(cluster_ids)

    best_group = None
    best_score = -1.0

    for g in all_run_groups:

        if not g.get("synthesis_comment"):
            continue

        gid_set = set(
            g["comment_ids"]
        )

        # Exact match = best possible
        if gid_set == target:

            return g["synthesis_comment"]

        intersection = len(
            gid_set & target
        )

        if intersection == 0:
            continue

        union_size = len(
            gid_set | target
        )

        jaccard = (
            intersection / union_size
        )

        if jaccard > best_score:

            best_score = jaccard
            best_group = g

    return (
        best_group["synthesis_comment"]
        if best_group
        else None
    )


# ============================================================
# 8. AGREEMENT SCORES
# ============================================================

def get_cluster_agreement(
    cluster_ids,
    pair_agreements
):

    agreements = []

    for i in range(len(cluster_ids)):

        for j in range(i + 1, len(cluster_ids)):

            pair = frozenset(
                [
                    cluster_ids[i],
                    cluster_ids[j]
                ]
            )

            if pair in pair_agreements:

                agreements.append(
                    pair_agreements[pair][
                        "agreement_ratio"
                    ]
                )

    return agreements


# ============================================================
# 9. BUILD FINAL CLUSTER ROWS
# ============================================================

def build_cluster_rows(
    patch_id,
    clusters,
    rows,
    all_run_groups,
    pair_agreements
):

    lookup = {
        r["comment_id"]: r
        for r in rows
    }

    cluster_rows = []

    for k, ids in enumerate(clusters):

        synthesis = (
            select_synthesis_for_cluster(
                ids,
                all_run_groups
            )
            if len(ids) > 1
            else None
        )

        cluster_rows.append(
            {
                "patch_id": patch_id,

                "num_comments": len(ids),

                "cluster_id":
                    f"{patch_id}_c{k}",

                "comment_ids": ids,

                "comments": [
                    lookup[i]["generated_comment"]
                    for i in ids
                ],

                "generation_systems": [
                    lookup[i]["generation_system"]
                    for i in ids
                ],

                "categories": [
                    lookup[i]["category"]
                    for i in ids
                ],

                "severities": [
                    lookup[i]["severity"]
                    for i in ids
                ],

                "synthesis_comment":
                    synthesis,

                "agreement_scores":
                    get_cluster_agreement(
                        ids,
                        pair_agreements
                    ),
            }
        )

    return cluster_rows


# ============================================================
# 10. PROCESS ONE PATCH FROM BATCH RESULTS
# ============================================================

def process_patch_from_batch(
    patch_id,
    rows,
    run_outputs
):
    """
    Reproduces the post-LLM processing of the original
    run_dedup_for_patch(), but WITHOUT making any API call.

    run_outputs must contain one LLM response per
    consistency run.
    """

    valid_ids = [
        r["comment_id"]
        for r in rows
    ]

    # --------------------------------------------------------
    # Singleton patch
    # --------------------------------------------------------

    if len(rows) <= 1:

        clusters = [
            [r["comment_id"]]
            for r in rows
        ]

        return build_cluster_rows(
            patch_id,
            clusters,
            rows,
            all_run_groups=[],
            pair_agreements={}
        )

    # --------------------------------------------------------
    # Process the 3 already-generated LLM outputs
    # --------------------------------------------------------

    run_pair_sets = []

    all_run_groups = []

    for run_idx, raw_text in enumerate(run_outputs):

        groups = parse_dedup_output(
            raw_text,
            valid_ids
        )

        if groups is None:

            print(
                f"[WARN] patch {patch_id} "
                f"run {run_idx}: "
                f"unparseable output"
            )

            groups = [
                {
                    "comment_ids": [i],
                    "synthesis_comment": None
                }
                for i in valid_ids
            ]

        run_pair_sets.append(
            groups_to_pairs(groups)
        )

        all_run_groups.extend(groups)

    # --------------------------------------------------------
    # Majority agreement
    # --------------------------------------------------------

    min_votes = (
        len(run_pair_sets) // 2 + 1
    )

    pair_counts = Counter()

    for pairs in run_pair_sets:

        for pair in pairs:

            pair_counts[pair] += 1

    pair_agreements = {

        pair: {
            "agreement_count": count,
            "agreement_ratio":
                count / len(run_pair_sets)
        }

        for pair, count
        in pair_counts.items()
    }

    # --------------------------------------------------------
    # Keep majority-agreed pairs
    # --------------------------------------------------------

    consistent_pairs = {

        pair

        for pair, info
        in pair_agreements.items()

        if info["agreement_count"]
        >= min_votes
    }

    # --------------------------------------------------------
    # Union-Find
    # --------------------------------------------------------

    uf = UnionFind(valid_ids)

    for pair in consistent_pairs:

        a, b = tuple(pair)

        uf.union(a, b)

    # --------------------------------------------------------
    # Build clusters
    # --------------------------------------------------------

    clusters_map = defaultdict(list)

    for comment_id in valid_ids:

        clusters_map[
            uf.find(comment_id)
        ].append(comment_id)

    clusters = list(
        clusters_map.values()
    )

    # --------------------------------------------------------
    # Final cluster rows
    # --------------------------------------------------------

    return build_cluster_rows(
        patch_id,
        clusters,
        rows,
        all_run_groups,
        pair_agreements
    )


# ============================================================
# 11. MAP BATCH RESULTS TO PATCHES / RUNS
# ============================================================

def build_batch_output_map(batch_results):
    """
    Converts the JSONL batch results into:

        {
            custom_id: llm_output
        }

    """

    output_map = {}

    for result in batch_results:

        custom_id = get_custom_id(
            result
        )

        if custom_id is None:
            print(
                "[WARN] Batch result has no custom_id"
            )
            continue

        output_map[
            custom_id
        ] = extract_batch_response(
            result
        )

    return output_map


# ============================================================
# 12. CUSTOM_ID -> PATCH + RUN
# ============================================================


# ============================================================
# 13. MAIN PIPELINE
# ============================================================
def run_batch_dedup_pipeline(
    df,
    batch_results_path
):
    """
    Post-process already completed batch results.

    The JSONL is assumed to be ordered as:

        patch 1 -> run 0
        patch 1 -> run 1
        patch 1 -> run 2

        patch 2 -> run 0
        patch 2 -> run 1
        patch 2 -> run 2

        ...

    No LLM/API calls are made here.
    """

    required_cols = [
        "comment_id",
        "generated_comment",
        "category",
        "generation_system",
        "patch_id",
        "severity",
        "hunk"
    ]

    for c in required_cols:

        if c not in df.columns:

            raise ValueError(
                f"Missing required column: {c}"
            )

    # --------------------------------------------------------
    # Load batch results
    # --------------------------------------------------------

    batch_results = load_batch_results(
        batch_results_path
    )

    batch_outputs = [
        extract_batch_response(result)
        for result in batch_results
    ]

    print(
        f"[BATCH] Loaded {len(batch_outputs)} outputs"
    )

    # --------------------------------------------------------
    # Group dataframe by patch
    # --------------------------------------------------------

    patch_groups = list(
        df.groupby("patch_id")
    )

    total_patches = len(patch_groups)

    expected_outputs = (
        total_patches *
        N_CONSISTENCY_RUNS
    )

    print(
        f"[INFO] Number of patches: "
        f"{total_patches}"
    )

    print(
        f"[INFO] Expected batch outputs: "
        f"{expected_outputs}"
    )

    print(
        f"[INFO] Actual batch outputs: "
        f"{len(batch_outputs)}"
    )

    if len(batch_outputs) != expected_outputs:

        raise ValueError(
            f"Number of batch results does not match "
            f"the expected number.\n"
            f"Expected: {expected_outputs} "
            f"({total_patches} patches × "
            f"{N_CONSISTENCY_RUNS} runs)\n"
            f"Found: {len(batch_outputs)}"
        )

    # --------------------------------------------------------
    # Process patches
    # --------------------------------------------------------

    results = []

    for patch_idx, (patch_id, group) in enumerate(
        patch_groups
    ):

        print(
            f"\n[PATCH {patch_idx + 1}/"
            f"{total_patches}] "
            f"{patch_id} "
            f"({len(group)} comments)"
        )

        rows = group.to_dict(
            "records"
        )

        for r in rows:

            r.setdefault(
                "generated_context",
                ""
            )

        # ----------------------------------------------------
        # Extract the 3 outputs corresponding to this patch
        # ----------------------------------------------------

        start_idx = (
            patch_idx *
            N_CONSISTENCY_RUNS
        )

        end_idx = (
            start_idx +
            N_CONSISTENCY_RUNS
        )

        run_outputs = batch_outputs[
            start_idx:end_idx
        ]

        print(
            f"  Using batch outputs "
            f"{start_idx} → {end_idx - 1}"
        )

        # ----------------------------------------------------
        # Process without API call
        # ----------------------------------------------------

        try:

            patch_cluster_rows = (
                process_patch_from_batch(
                    patch_id,
                    rows,
                    run_outputs
                )
            )

        except Exception as e:

            print(
                f"[FAIL] patch {patch_id}: {e}"
            )

            patch_cluster_rows = [

                {
                    "patch_id":
                        patch_id,

                    "num_comments":
                        len(rows),

                    "cluster_id":
                        f"{patch_id}_c{k}",

                    "comment_ids":
                        [r["comment_id"]],

                    "comments":
                        [r["generated_comment"]],

                    "generation_systems":
                        [r["generation_system"]],

                    "categories":
                        [r["category"]],

                    "severities":
                        [r["severity"]],

                    "synthesis_comment":
                        None,

                    "error":
                        str(e),
                }

                for k, r in enumerate(rows)
            ]

        results.extend(
            patch_cluster_rows
        )

    # --------------------------------------------------------
    # Final DataFrame
    # --------------------------------------------------------

    final_df = pd.DataFrame(
        results
    )

    print(
        "\n[DONE] Batch post-processing finished"
    )

    print(
        f"[DONE] Final dataframe shape: "
        f"{final_df.shape}"
    )

    return final_df

# ============================================================
# 14. RUN
# ============================================================

# Your original dataframe should already be loaded as `df`.
final_df = run_batch_dedup_pipeline(
    df=df,
    batch_results_path=BATCH_RESULTS_PATH
)

print(final_df.head())

[BATCH] Loaded 1512 batch results
[BATCH] Loaded 1512 outputs
[INFO] Number of patches: 504
[INFO] Expected batch outputs: 1512
[INFO] Actual batch outputs: 1512

[PATCH 1/504] P000000 (2 comments)
  Using batch outputs 0 → 2

[PATCH 2/504] P000001 (9 comments)
  Using batch outputs 3 → 5

[PATCH 3/504] P000002 (3 comments)
  Using batch outputs 6 → 8

[PATCH 4/504] P000003 (6 comments)
  Using batch outputs 9 → 11

[PATCH 5/504] P000004 (8 comments)
  Using batch outputs 12 → 14

[PATCH 6/504] P000005 (6 comments)
  Using batch outputs 15 → 17

[PATCH 7/504] P000006 (10 comments)
  Using batch outputs 18 → 20

[PATCH 8/504] P000007 (3 comments)
  Using batch outputs 21 → 23

[PATCH 9/504] P000008 (6 comments)
  Using batch outputs 24 → 26

[PATCH 10/504] P000009 (10 comments)
  Using batch outputs 27 → 29

[PATCH 11/504] P000010 (8 comments)
  Using batch outputs 30 → 32

[PATCH 12/504] P000011 (4 comments)
  Using batch outputs 33 → 35

[PATCH 13/504] P000012 (5 comments)
  Using bat

In [17]:
final_df.to_csv('../../data/df_n4.csv', index=False)